# 5장 Release Decision 검토

## Goal

데이터, 모델, serving과 운영 증거를 하나의 release 판단으로 연결합니다. Candidate A는 배포하지 않고 Candidate B만 승인 overlay에 포함되는지 확인합니다.

In [1]:
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").is_file(), "tta-aiqa repository root를 찾지 못했습니다."
ROOT

PosixPath('/Users/seungbaeji/Workspace/tta/tta-aiqa')

In [2]:
import json
from pathlib import Path
import pandas as pd

canonical_path = ROOT / "reference/evidence/model/revisions/v2/canonical-benchmark.json"
canonical = json.loads(canonical_path.read_text(encoding="utf-8"))
decisions = pd.DataFrame(canonical["decisions"]).set_index("profile")
decisions[["decision", "checks"]]

,decision,checks
profile,,
candidate-a,HOLD,"{'false_negative_reduction': False, 'pr_auc_vs..."
candidate-b,APPROVE,"{'false_negative_reduction': True, 'pr_auc_vs_..."


## Steps

### 1. 배포 overlay와 승인 결과 연결

In [3]:
overlay_root = ROOT / "deploy/kubernetes/overlays"
overlays = {
    name: "\n".join(path.read_text(encoding="utf-8") for path in sorted((overlay_root / name).glob("*.yaml")))
    for name in ("baseline", "candidate-b", "rollback")
}
summary = pd.DataFrame([
    {
        "overlay": name,
        "contains_baseline": "baseline-f2576f12512a" in document,
        "contains_candidate_b": "candidate-b-c712a8e52344" in document,
        "contains_candidate_a": "candidate-a" in document,
    }
    for name, document in overlays.items()
]).set_index("overlay")
summary

,contains_baseline,contains_candidate_b,contains_candidate_a
overlay,,,
baseline,True,False,False
candidate-b,False,True,False
rollback,True,False,False


### 2. 판단 기록 구성

In [4]:
release_record = {
    "dataset_sha256": canonical["sealed_test"]["dataset_sha256"],
    "sealed_test_status": canonical["sealed_test"]["status"],
    "candidate_a": decisions.loc["candidate-a", "decision"],
    "candidate_b": decisions.loc["candidate-b", "decision"],
    "deployment_allowed": canonical["deployment_allowed"],
    "approved_overlay": "candidate-b",
    "rollback_overlay": "rollback",
}
release_record

{'dataset_sha256': '051491069cd385789047031d87ad92f3ebf86868f7b2da410307b639b906aa77',
 'sealed_test_status': 'evaluated_once',
 'candidate_a': 'HOLD',
 'candidate_b': 'APPROVE',
 'deployment_allowed': True,
 'approved_overlay': 'candidate-b',
 'rollback_overlay': 'rollback'}

## Checks

In [5]:
assert release_record["candidate_a"] == "HOLD"
assert release_record["candidate_b"] == "APPROVE"
assert release_record["deployment_allowed"] is True
assert summary.loc["candidate-b", "contains_candidate_b"]
assert not summary["contains_candidate_a"].any()
assert summary.loc["rollback", "contains_baseline"]
print("Release decision checks passed.")

Release decision checks passed.


## Next Steps

Argo CD에서 Candidate B overlay를 sync하고 같은 Risk API URL의 model version과 Grafana telemetry를 확인합니다. 실습 종료 후 rollback overlay로 baseline 복구를 검증합니다.